# 4. Scoreline Probability Model

Goal: The predicted home and away goal distributions are combined into a full scoreline probability matrix. From this matrix we derive match-level probabilities such as win/draw/loss, BTTS, and totals.

Note: The important thing isn't feature engineering and realy max optimization of the models (since we already did that for the 1st model in notebook 03), but rather get an overview of them

In [ ]:
## 4.1 Import Libraries

import sys
sys.path.append("../src")

import numpy as np
import pandas as pd
import joblib

from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

from Final_Data_Loader import load_clean_match_data
from Final_Feature_Engineering import build_feature_dataset
from Final_Model_Preperation import prepare_modeling_data


In [ ]:
## 4.2 Load Goal Model Predictions

SEASONS = [
    "93-94", "94-95", "95-96", "96-97", "97-98",
    "98-99", "99-00", "00-01",
    "01-02", "02-03", "03-04", "04-05", "05-06",
    "06-07", "07-08", "08-09", "09-10", "10-11",
    "11-12", "12-13", "13-14", "14-15", "15-16",
    "16-17", "17-18", "18-19", "19-20", "20-21",
    "21-22", "22-23", "23-24", "24-25", "25-26"
]

DATA_DIR = "../data/raw"

TRAIN_END = "2019-07-01"
VAL_END = "2021-07-01"


In [ ]:
print("Loading raw data...")
df = load_clean_match_data(
    seasons=SEASONS,
    data_dir=DATA_DIR,
    filename_template="EPL {season}.csv",
    drop_betting=True,
    verbose=True,
)

print("Raw dataframe shape:", df.shape)
print("Raw date range:", df["Date"].min(), "to", df["Date"].max())


Loading raw data...
Loading ../data/raw/EPL 93-94.csv ...
Loading ../data/raw/EPL 94-95.csv ...
Loading ../data/raw/EPL 95-96.csv ...
Loading ../data/raw/EPL 96-97.csv ...
Loading ../data/raw/EPL 97-98.csv ...
Loading ../data/raw/EPL 98-99.csv ...
Loading ../data/raw/EPL 99-00.csv ...
Loading ../data/raw/EPL 00-01.csv ...
Loading ../data/raw/EPL 01-02.csv ...
Loading ../data/raw/EPL 02-03.csv ...
Loading ../data/raw/EPL 03-04.csv ...
Loading ../data/raw/EPL 04-05.csv ...
Loading ../data/raw/EPL 05-06.csv ...
Loading ../data/raw/EPL 06-07.csv ...
Loading ../data/raw/EPL 07-08.csv ...
Loading ../data/raw/EPL 08-09.csv ...
Loading ../data/raw/EPL 09-10.csv ...
Loading ../data/raw/EPL 10-11.csv ...
Loading ../data/raw/EPL 11-12.csv ...
Loading ../data/raw/EPL 12-13.csv ...
Loading ../data/raw/EPL 13-14.csv ...
Loading ../data/raw/EPL 14-15.csv ...
Loading ../data/raw/EPL 15-16.csv ...
Loading ../data/raw/EPL 16-17.csv ...
Loading ../data/raw/EPL 17-18.csv ...
Loading ../data/raw/EPL 18-19.

c:\angelo\project EPL predictor\MatchMind-main\notebooks\../src\Final_Data_Loader.py:66: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df["Date"] = pd.to_datetime(df["Date"], dayfirst=True, errors="coerce")


Rows with missing Date after parsing: 697
Raw dataframe shape: (12474, 104)
Raw date range: 1993-08-14 00:00:00 to 2026-02-02 00:00:00


In [ ]:
## 4.3 BUILD FEATURE DATASET

print("\nBuilding feature dataset...")
match_df = build_feature_dataset(df)

print("Feature dataset shape:", match_df.shape)
print("Feature date range:", match_df["Date"].min(), "to", match_df["Date"].max())




Building feature dataset...
Feature dataset shape: (12474, 155)
Feature date range: 1993-08-14 00:00:00 to 2026-02-02 00:00:00


In [ ]:
# 4.4 GET FINAL FEATURE COLUMNS USING MY EXISTING PREP FUNCTION

print("\nPreparing base modeling data to recover final feature columns...")
data = prepare_modeling_data(
    match_df,
    train_end=TRAIN_END,
    val_end=VAL_END,
    target_col="Result",
)

feature_cols = data["feature_cols"]

print("Number of final feature columns:", len(feature_cols))



Preparing base modeling data to recover final feature columns...
Number of final feature columns: 126


In [ ]:

## 4.5 DATE-BASED SPLITS FOR GOAL MODELS

match_df = match_df.sort_values("Date").reset_index(drop=True)

train_mask = match_df["Date"] < TRAIN_END
val_mask = (match_df["Date"] >= TRAIN_END) & (match_df["Date"] < VAL_END)
test_mask = match_df["Date"] >= VAL_END

train_df = match_df.loc[train_mask].copy()
val_df = match_df.loc[val_mask].copy()
test_df = match_df.loc[test_mask].copy()

print("Train rows:", len(train_df))
print("Val rows:", len(val_df))
print("Test rows:", len(test_df))



Train rows: 9954
Val rows: 760
Test rows: 1760


In [ ]:
## 4.6 BUILD X AND y FOR GOAL PREDICTION

X_train = train_df[feature_cols].replace([np.inf, -np.inf], np.nan)
X_val = val_df[feature_cols].replace([np.inf, -np.inf], np.nan)
X_test = test_df[feature_cols].replace([np.inf, -np.inf], np.nan)

y_train_home = train_df["GF_home"].astype(float)
y_val_home = val_df["GF_home"].astype(float)
y_test_home = test_df["GF_home"].astype(float)

y_train_away = train_df["GF_away"].astype(float)
y_val_away = val_df["GF_away"].astype(float)
y_test_away = test_df["GF_away"].astype(float)

print("X_train shape:", X_train.shape)
print("X_val shape:", X_val.shape)
print("X_test shape:", X_test.shape)

print("Home goals target mean (train):", y_train_home.mean())
print("Away goals target mean (train):", y_train_away.mean())

X_train shape: (9954, 126)
X_val shape: (760, 126)
X_test shape: (1760, 126)
Home goals target mean (train): 1.5243118344384168
Away goals target mean (train): 1.1262808921036769


In [ ]:
## 4.7 HOME GOALS MODEL and optimization

home_goals_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=800,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=25,
    subsample=0.7,
    colsample_bytree=0.7,
    gamma=0,
    random_state=42,
    eval_metric="rmse",
    early_stopping_rounds=50
)

home_goals_model.fit(
    X_train,
    y_train_home,
    eval_set=[(X_val, y_val_home)],
    verbose=False
)

home_val_pred = home_goals_model.predict(X_val)
home_test_pred = home_goals_model.predict(X_test)

print("HOME GOALS - VAL MAE:", mean_absolute_error(y_val_home, home_val_pred))
print("HOME GOALS - VAL RMSE:", root_mean_squared_error(y_val_home, home_val_pred))
print("HOME GOALS - VAL R2:", r2_score(y_val_home, home_val_pred))

print("HOME GOALS - TEST MAE:", mean_absolute_error(y_test_home, home_test_pred))
print("HOME GOALS - TEST RMSE:", root_mean_squared_error(y_test_home, home_test_pred))
print("HOME GOALS - TEST R2:", r2_score(y_test_home, home_test_pred))

HOME GOALS - VAL MAE: 0.9443698755220363
HOME GOALS - VAL RMSE: 1.204990129642264
HOME GOALS - VAL R2: 0.12189728786474929
HOME GOALS - TEST MAE: 0.9831476238302209
HOME GOALS - TEST RMSE: 1.2444826226331656
HOME GOALS - TEST R2: 0.1250884099758044


In [10]:
print("Predicted mean:", home_test_pred.mean())
print("Actual mean:", y_test_home.mean())

Predicted mean: 1.556488
Actual mean: 1.6079545454545454


In [ ]:
## 4.8 AWAY GOALS MODEL

away_goals_model = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=800,
    max_depth=4,
    learning_rate=0.03,
    min_child_weight=9,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    random_state=42,
    eval_metric="rmse",
    early_stopping_rounds=50
)

away_goals_model.fit(
    X_train,
    y_train_away,
    eval_set=[(X_val, y_val_away)],
    verbose=False
)

away_val_pred = away_goals_model.predict(X_val)
away_test_pred = away_goals_model.predict(X_test)

print("AWAY GOALS - VAL MAE:", mean_absolute_error(y_val_away, away_val_pred))
print("AWAY GOALS - VAL RMSE:", root_mean_squared_error(y_val_away, away_val_pred))
print("AWAY GOALS - VAL R2:", r2_score(y_val_away, away_val_pred))

print("AWAY GOALS - TEST MAE:", mean_absolute_error(y_test_away, away_test_pred))
print("AWAY GOALS - TEST RMSE:", root_mean_squared_error(y_test_away, away_test_pred))
print("AWAY GOALS - TEST R2:", r2_score(y_test_away, away_test_pred))

AWAY GOALS - VAL MAE: 0.8805135923388757
AWAY GOALS - VAL RMSE: 1.1636387808958084
AWAY GOALS - VAL R2: 0.10442474180009409
AWAY GOALS - TEST MAE: 0.8880149495025927
AWAY GOALS - TEST RMSE: 1.1479951237171615
AWAY GOALS - TEST R2: 0.1064615278732346


In [ ]:
#I changed my mine i won't be using regression I will make the model predict integer values and there probability but I won't use the logloss function instwad i will search if there is a function that penalizes more if it is more away but not like regression 

In [ ]:
## 4.9 HOME GOALS MODEL (multiclass classification with logglos)

import numpy as np
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error, root_mean_squared_error

def cap_goals(x, cap=5):
    return min(int(x), cap)

# Cap targets: classes 0,1,2,3,4,5 where 5 means 5+
y_train_home_cls = y_train_home.apply(cap_goals)
y_val_home_cls   = y_val_home.apply(cap_goals)
y_test_home_cls  = y_test_home.apply(cap_goals)

home_goals_model = XGBClassifier(
    objective="multi:softprob",
    num_class=6,
    n_estimators=6000,
    max_depth=3,
    learning_rate=0.05,
    min_child_weight=25,
    subsample=0.7,
    colsample_bytree=0.7,
    gamma=0,
    random_state=42,
    eval_metric="mlogloss",
    early_stopping_rounds=50
)

home_goals_model.fit(
    X_train,
    y_train_home_cls,
    eval_set=[(X_val, y_val_home_cls)],
    verbose=False
)

# Predicted class probabilities
home_val_proba = home_goals_model.predict_proba(X_val)   # shape: (n_samples, 6)
home_test_proba = home_goals_model.predict_proba(X_test)

# Most likely integer class
home_val_pred_class = home_val_proba.argmax(axis=1)
home_test_pred_class = home_test_proba.argmax(axis=1)

# Expected goals from probabilities
goal_values = np.array([0, 1, 2, 3, 4, 5], dtype=float)
home_val_pred_exp = (home_val_proba * goal_values).sum(axis=1)
home_test_pred_exp = (home_test_proba * goal_values).sum(axis=1)

print("HOME GOALS CLASSIFICATION - VAL ACC:", accuracy_score(y_val_home_cls, home_val_pred_class))
print("HOME GOALS CLASSIFICATION - VAL LOGLOSS:", log_loss(y_val_home_cls, home_val_proba))

print("HOME GOALS CLASSIFICATION - TEST ACC:", accuracy_score(y_test_home_cls, home_test_pred_class))
print("HOME GOALS CLASSIFICATION - TEST LOGLOSS:", log_loss(y_test_home_cls, home_test_proba))

# Fair comparison to regression: compare expected goals vs true goals
print("HOME GOALS EXPECTED - VAL MAE:", mean_absolute_error(y_val_home, home_val_pred_exp))
print("HOME GOALS EXPECTED - VAL RMSE:", root_mean_squared_error(y_val_home, home_val_pred_exp))

print("HOME GOALS EXPECTED - TEST MAE:", mean_absolute_error(y_test_home, home_test_pred_exp))
print("HOME GOALS EXPECTED - TEST RMSE:", root_mean_squared_error(y_test_home, home_test_pred_exp))

HOME GOALS CLASSIFICATION - VAL ACC: 0.3605263157894737
HOME GOALS CLASSIFICATION - VAL LOGLOSS: 1.4608064655403215
HOME GOALS CLASSIFICATION - TEST ACC: 0.33238636363636365
HOME GOALS CLASSIFICATION - TEST LOGLOSS: 1.524996849941567
HOME GOALS EXPECTED - VAL MAE: 0.9471875368093606
HOME GOALS EXPECTED - VAL RMSE: 1.2072942030768723
HOME GOALS EXPECTED - TEST MAE: 0.9831414946751796
HOME GOALS EXPECTED - TEST RMSE: 1.245083138851347


In [ ]:
# about the same as regression the other options are poisson (discarded because the goals aren't exactly poisson)
# I will use an ordinal modality loss function that penalizes more the further away the prediction is from the true value but not like regression

In [ ]:

## 4.10 ORDINAL XGBOOST FOR HOME GOALS: P(HG > k)
# Classes: 0, 1, 2, 3, 4, 5+  -> thresholds k = 0,1,2,3,4
import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.metrics import log_loss, accuracy_score, mean_absolute_error, root_mean_squared_error

# ---------------------------------------------------------
#  1. Helper functions
# ---------------------------------------------------------

def cap_goals(x, cap=5):
    return min(int(x), cap)

def make_ordinal_targets(y, max_class=5):
    """
    For classes 0,1,2,3,4,5, create binary targets:
    y_gt_0, y_gt_1, y_gt_2, y_gt_3, y_gt_4
    where y_gt_k = 1 if y > k else 0
    """
    y = pd.Series(y).astype(int).clip(lower=0, upper=max_class)
    ordinal = {}
    for k in range(max_class):
        ordinal[f"gt_{k}"] = (y > k).astype(int)
    return ordinal

def ordinal_probs_from_thresholds(threshold_probs):
    """
    Convert threshold probabilities into class probabilities.

    Input:
        threshold_probs: array of shape (n_samples, 5)
        columns are:
            P(y > 0), P(y > 1), P(y > 2), P(y > 3), P(y > 4)

    Output:
        class_probs: array of shape (n_samples, 6)
        columns are:
            P(y=0), P(y=1), P(y=2), P(y=3), P(y=4), P(y=5)
        where class 5 means 5+
    """
    p_gt_0 = threshold_probs[:, 0]
    p_gt_1 = threshold_probs[:, 1]
    p_gt_2 = threshold_probs[:, 2]
    p_gt_3 = threshold_probs[:, 3]
    p_gt_4 = threshold_probs[:, 4]

    p0 = 1.0 - p_gt_0
    p1 = p_gt_0 - p_gt_1
    p2 = p_gt_1 - p_gt_2
    p3 = p_gt_2 - p_gt_3
    p4 = p_gt_3 - p_gt_4
    p5 = p_gt_4

    class_probs = np.column_stack([p0, p1, p2, p3, p4, p5])

    # Numerical cleanup
    class_probs = np.clip(class_probs, 0.0, 1.0)
    row_sums = class_probs.sum(axis=1, keepdims=True)
    row_sums = np.where(row_sums == 0, 1.0, row_sums)
    class_probs = class_probs / row_sums

    return class_probs

def enforce_monotonic_thresholds(threshold_probs):
    """
    Ensure:
        P(y > 0) >= P(y > 1) >= ... >= P(y > 4)

    This can be violated slightly because each binary model is trained separately.
    We fix it by sweeping left to right.
    """
    fixed = threshold_probs.copy()
    for j in range(1, fixed.shape[1]):
        fixed[:, j] = np.minimum(fixed[:, j - 1], fixed[:, j])
    fixed = np.clip(fixed, 0.0, 1.0)
    return fixed

def ranked_probability_score(y_true, class_probs, n_classes=6):
    """
    RPS for ordered classes.
    Lower is better.
    """
    y_true = np.asarray(y_true).astype(int)
    class_probs = np.asarray(class_probs)

    true_one_hot = np.eye(n_classes)[y_true]
    cum_pred = np.cumsum(class_probs, axis=1)
    cum_true = np.cumsum(true_one_hot, axis=1)

    return np.mean(np.sum((cum_pred - cum_true) ** 2, axis=1) / (n_classes - 1))


# ---------------------------------------------------------
# 2. Cap targets to 0,1,2,3,4,5(=5+)
# ---------------------------------------------------------

y_train_home_ord = y_train_home.apply(cap_goals)
y_val_home_ord   = y_val_home.apply(cap_goals)
y_test_home_ord  = y_test_home.apply(cap_goals)

# Build threshold targets
train_targets = make_ordinal_targets(y_train_home_ord, max_class=5)
val_targets   = make_ordinal_targets(y_val_home_ord, max_class=5)
test_targets  = make_ordinal_targets(y_test_home_ord, max_class=5)

# ---------------------------------------------------------
# 3. Train 5 binary XGBoost models: P(HG > k)
# ---------------------------------------------------------

home_ordinal_models = {}
val_threshold_probs = []
test_threshold_probs = []

for k in range(5):
    print(f"Training model for P(HG > {k})")

    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=800,
        max_depth=3,
        learning_rate=0.05,
        min_child_weight=25,
        subsample=0.7,
        colsample_bytree=0.7,
        gamma=0,
        random_state=42,
        eval_metric="logloss",
        early_stopping_rounds=50
    )

    y_train_k = train_targets[f"gt_{k}"]
    y_val_k   = val_targets[f"gt_{k}"]

    model.fit(
        X_train,
        y_train_k,
        eval_set=[(X_val, y_val_k)],
        verbose=False
    )

    home_ordinal_models[k] = model

    # Probability of class 1 = P(HG > k)
    val_prob_k = model.predict_proba(X_val)[:, 1]
    test_prob_k = model.predict_proba(X_test)[:, 1]

    val_threshold_probs.append(val_prob_k)
    test_threshold_probs.append(test_prob_k)

# Shape: (n_samples, 5)
val_threshold_probs = np.column_stack(val_threshold_probs)
test_threshold_probs = np.column_stack(test_threshold_probs)

# ---------------------------------------------------------
# 4. Enforce monotonic thresholds
# ---------------------------------------------------------

val_threshold_probs = enforce_monotonic_thresholds(val_threshold_probs)
test_threshold_probs = enforce_monotonic_thresholds(test_threshold_probs)

# ---------------------------------------------------------
# 5. Convert threshold probs -> class probs
# ---------------------------------------------------------

home_val_proba = ordinal_probs_from_thresholds(val_threshold_probs)
home_test_proba = ordinal_probs_from_thresholds(test_threshold_probs)

# Most likely class
home_val_pred_class = np.argmax(home_val_proba, axis=1)
home_test_pred_class = np.argmax(home_test_proba, axis=1)

# Expected goals from class probabilities
goal_values = np.array([0, 1, 2, 3, 4, 5], dtype=float)
home_val_pred_exp = (home_val_proba * goal_values).sum(axis=1)
home_test_pred_exp = (home_test_proba * goal_values).sum(axis=1)

# ---------------------------------------------------------
# 6. Evaluation
# ---------------------------------------------------------

print("\n===== HOME GOALS ORDINAL MODEL =====")

print("VAL ACCURACY:", accuracy_score(y_val_home_ord, home_val_pred_class))
print("TEST ACCURACY:", accuracy_score(y_test_home_ord, home_test_pred_class))

print("VAL LOGLOSS:", log_loss(y_val_home_ord, home_val_proba, labels=[0, 1, 2, 3, 4, 5]))
print("TEST LOGLOSS:", log_loss(y_test_home_ord, home_test_proba, labels=[0, 1, 2, 3, 4, 5]))

print("VAL RPS:", ranked_probability_score(y_val_home_ord, home_val_proba))
print("TEST RPS:", ranked_probability_score(y_test_home_ord, home_test_proba))

print("VAL EXPECTED-GOALS MAE:", mean_absolute_error(y_val_home, home_val_pred_exp))
print("TEST EXPECTED-GOALS MAE:", mean_absolute_error(y_test_home, home_test_pred_exp))

print("VAL EXPECTED-GOALS RMSE:", root_mean_squared_error(y_val_home, home_val_pred_exp))
print("TEST EXPECTED-GOALS RMSE:", root_mean_squared_error(y_test_home, home_test_pred_exp))

# ---------------------------------------------------------
# 7. Example: inspect first few predicted distributions
# ---------------------------------------------------------

pred_dist_df = pd.DataFrame(
    home_test_proba,
    columns=["P0", "P1", "P2", "P3", "P4", "P5plus"]
)

pred_dist_df["pred_class"] = home_test_pred_class
pred_dist_df["pred_xg"] = home_test_pred_exp
pred_dist_df["true_goals_capped"] = y_test_home_ord.values
pred_dist_df["true_goals_raw"] = y_test_home.values

print("\nSample predictions:")
print(pred_dist_df.head(10))

Training model for P(HG > 0)
Training model for P(HG > 1)
Training model for P(HG > 2)
Training model for P(HG > 3)
Training model for P(HG > 4)

===== HOME GOALS ORDINAL MODEL =====
VAL ACCURACY: 0.3513157894736842
TEST ACCURACY: 0.31988636363636364
VAL LOGLOSS: 1.4553658666818594
TEST LOGLOSS: 1.5446931982082202
VAL RPS: 0.12352693219973793
TEST RPS: 0.13081472511497966
VAL EXPECTED-GOALS MAE: 0.9425189579276393
TEST EXPECTED-GOALS MAE: 0.980386132879225
VAL EXPECTED-GOALS RMSE: 1.2028443826067865
TEST EXPECTED-GOALS RMSE: 1.2431892327504548

Sample predictions:
         P0        P1        P2        P3        P4    P5plus  pred_class  \
0  0.330872  0.368846  0.185980  0.087391  0.019272  0.007639           1   
1  0.317060  0.335231  0.193142  0.106958  0.038723  0.008886           1   
2  0.191704  0.301803  0.243798  0.150427  0.085515  0.026753           1   
3  0.121082  0.246585  0.325218  0.146284  0.119452  0.041380           2   
4  0.313650  0.338641  0.208634  0.101760  0

In [ ]:
#slightly better so I will go with this one and try to optimize it more and then I will do the same for away goals

In [23]:
# usefull functions
# =========================================================
# TUNING ORDINAL XGBOOST
# Classes: 0,1,2,3,4,5+
# Models: P(HG>0), P(HG>1), P(HG>2), P(HG>3), P(HG>4)
# =========================================================

import numpy as np
import pandas as pd
from itertools import product
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error, mean_squared_error

# ---------------------------------------------------------
# Helper functions
# ---------------------------------------------------------

def cap_goals(x, cap=5):
    return min(int(x), cap)

def make_ordinal_targets(y, max_class=5):
    y = pd.Series(y).astype(int).clip(lower=0, upper=max_class)
    ordinal = {}
    for k in range(max_class):
        ordinal[f"gt_{k}"] = (y > k).astype(int)
    return ordinal

def enforce_monotonic_thresholds(threshold_probs):
    fixed = threshold_probs.copy()
    for j in range(1, fixed.shape[1]):
        fixed[:, j] = np.minimum(fixed[:, j - 1], fixed[:, j])
    return np.clip(fixed, 0.0, 1.0)

def ordinal_probs_from_thresholds(threshold_probs):
    p_gt_0 = threshold_probs[:, 0]
    p_gt_1 = threshold_probs[:, 1]
    p_gt_2 = threshold_probs[:, 2]
    p_gt_3 = threshold_probs[:, 3]
    p_gt_4 = threshold_probs[:, 4]

    p0 = 1.0 - p_gt_0
    p1 = p_gt_0 - p_gt_1
    p2 = p_gt_1 - p_gt_2
    p3 = p_gt_2 - p_gt_3
    p4 = p_gt_3 - p_gt_4
    p5 = p_gt_4

    class_probs = np.column_stack([p0, p1, p2, p3, p4, p5])
    class_probs = np.clip(class_probs, 0.0, 1.0)

    row_sums = class_probs.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1.0
    class_probs = class_probs / row_sums

    return class_probs

def ranked_probability_score(y_true, class_probs, n_classes=6):
    y_true = np.asarray(y_true).astype(int)
    class_probs = np.asarray(class_probs)

    true_one_hot = np.eye(n_classes)[y_true]
    cum_pred = np.cumsum(class_probs, axis=1)
    cum_true = np.cumsum(true_one_hot, axis=1)

    return np.mean(np.sum((cum_pred - cum_true) ** 2, axis=1) / (n_classes - 1))

def rmse(y_true, y_pred):
    return np.sqrt(root_mean_squared_error(y_true, y_pred))

In [24]:
# ---------------------------------------------------------
# Prepare capped ordinal targets
# ---------------------------------------------------------

y_train_ord = pd.Series(y_train_home).apply(cap_goals)
y_val_ord   = pd.Series(y_val_home).apply(cap_goals)

train_targets = make_ordinal_targets(y_train_ord, max_class=5)
val_targets   = make_ordinal_targets(y_val_ord, max_class=5)

In [25]:
from itertools import product
from sklearn.metrics import accuracy_score, log_loss, mean_absolute_error, mean_squared_error

In [26]:
# ---------------------------------------------------------
# Parameter grid - choose manually
# ---------------------------------------------------------

n_estimators_list = [6000]
max_depth_list = [2,4,6,8]
learning_rate_list = [0.05]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [3,5,7,9,15,20,25,30]
gamma_list = [0,2,5]

results = []

best_rps = float("inf")
best_params = None
best_models = None
best_val_proba = None

total = (
    len(n_estimators_list)
    * len(max_depth_list)
    * len(learning_rate_list)
    * len(min_child_weight_list)
    * len(gamma_list)
    * len(subsample_list)
    * len(colsample_list)
)

run = 0

for n_est, max_depth, lr, mcw, g, s, c in product(
    n_estimators_list,
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    models = {}
    val_threshold_probs = []

    # Train 5 binary models
    for k in range(5):
        clf = XGBClassifier(
            objective="binary:logistic",
            n_estimators=n_est,
            max_depth=max_depth,
            learning_rate=lr,
            subsample=s,
            colsample_bytree=c,
            min_child_weight=mcw,
            gamma=g,
            random_state=42,
            eval_metric="logloss",
            early_stopping_rounds=50,
            tree_method="hist"
        )

        clf.fit(
            X_train,
            train_targets[f"gt_{k}"],
            eval_set=[(X_val, val_targets[f"gt_{k}"])],
            verbose=False
        )

        models[k] = clf
        val_threshold_probs.append(clf.predict_proba(X_val)[:, 1])

    # Stack threshold probs
    val_threshold_probs = np.column_stack(val_threshold_probs)

    # Fix monotonicity and convert to class probs
    val_threshold_probs = enforce_monotonic_thresholds(val_threshold_probs)
    val_proba = ordinal_probs_from_thresholds(val_threshold_probs)

    # Predictions
    val_pred = np.argmax(val_proba, axis=1)
    goal_values = np.array([0, 1, 2, 3, 4, 5], dtype=float)
    val_xg = (val_proba * goal_values).sum(axis=1)

    # Metrics
    acc = accuracy_score(y_val_ord, val_pred)
    ll = log_loss(y_val_ord, val_proba, labels=[0, 1, 2, 3, 4, 5])
    rps = ranked_probability_score(y_val_ord, val_proba)
    mae_xg = mean_absolute_error(y_val_home, val_xg)
    rmse_xg = rmse(y_val_home, val_xg)

    results.append({
        "n_estimators": n_est,
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll,
        "val_rps": rps,
        "val_xg_mae": mae_xg,
        "val_xg_rmse": rmse_xg
    })

    print(
        f"[{run}/{total}] "
        f"trees={n_est} depth={max_depth} lr={lr} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}, rps={rps:.4f}, xg_mae={mae_xg:.4f}"
    )

    if rps < best_rps:
        best_rps = rps
        best_params = {
            "n_estimators": n_est,
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_models = models
        best_val_proba = val_proba


results_df = pd.DataFrame(results).sort_values(["val_rps", "val_logloss", "val_xg_mae"])

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))

[1/96] trees=6000 depth=2 lr=0.05 mcw=3 gamma=0 subsample=0.8 colsample=0.8 -> acc=0.3579, logloss=1.4485, rps=0.1233, xg_mae=0.9418
[2/96] trees=6000 depth=2 lr=0.05 mcw=3 gamma=2 subsample=0.8 colsample=0.8 -> acc=0.3592, logloss=1.4481, rps=0.1233, xg_mae=0.9420
[3/96] trees=6000 depth=2 lr=0.05 mcw=3 gamma=5 subsample=0.8 colsample=0.8 -> acc=0.3592, logloss=1.4480, rps=0.1233, xg_mae=0.9422
[4/96] trees=6000 depth=2 lr=0.05 mcw=5 gamma=0 subsample=0.8 colsample=0.8 -> acc=0.3632, logloss=1.4470, rps=0.1232, xg_mae=0.9415
[5/96] trees=6000 depth=2 lr=0.05 mcw=5 gamma=2 subsample=0.8 colsample=0.8 -> acc=0.3632, logloss=1.4451, rps=0.1232, xg_mae=0.9418
[6/96] trees=6000 depth=2 lr=0.05 mcw=5 gamma=5 subsample=0.8 colsample=0.8 -> acc=0.3605, logloss=1.4463, rps=0.1233, xg_mae=0.9418
[7/96] trees=6000 depth=2 lr=0.05 mcw=7 gamma=0 subsample=0.8 colsample=0.8 -> acc=0.3553, logloss=1.4484, rps=0.1233, xg_mae=0.9412
[8/96] trees=6000 depth=2 lr=0.05 mcw=7 gamma=2 subsample=0.8 colsamp

In [28]:

n_estimators_list = [6000]
max_depth_list = [2,3]
learning_rate_list = [0.05]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [15,20,25,30]
gamma_list = [4,6,8,10]

results = []

best_rps = float("inf")
best_params = None
best_models = None
best_val_proba = None

total = (
    len(n_estimators_list)
    * len(max_depth_list)
    * len(learning_rate_list)
    * len(min_child_weight_list)
    * len(gamma_list)
    * len(subsample_list)
    * len(colsample_list)
)

run = 0

for n_est, max_depth, lr, mcw, g, s, c in product(
    n_estimators_list,
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    models = {}
    val_threshold_probs = []

    # Train 5 binary models
    for k in range(5):
        clf = XGBClassifier(
            objective="binary:logistic",
            n_estimators=n_est,
            max_depth=max_depth,
            learning_rate=lr,
            subsample=s,
            colsample_bytree=c,
            min_child_weight=mcw,
            gamma=g,
            random_state=42,
            eval_metric="logloss",
            early_stopping_rounds=50,
            tree_method="hist"
        )

        clf.fit(
            X_train,
            train_targets[f"gt_{k}"],
            eval_set=[(X_val, val_targets[f"gt_{k}"])],
            verbose=False
        )

        models[k] = clf
        val_threshold_probs.append(clf.predict_proba(X_val)[:, 1])

    # Stack threshold probs
    val_threshold_probs = np.column_stack(val_threshold_probs)

    # Fix monotonicity and convert to class probs
    val_threshold_probs = enforce_monotonic_thresholds(val_threshold_probs)
    val_proba = ordinal_probs_from_thresholds(val_threshold_probs)

    # Predictions
    val_pred = np.argmax(val_proba, axis=1)
    goal_values = np.array([0, 1, 2, 3, 4, 5], dtype=float)
    val_xg = (val_proba * goal_values).sum(axis=1)

    # Metrics
    acc = accuracy_score(y_val_ord, val_pred)
    ll = log_loss(y_val_ord, val_proba, labels=[0, 1, 2, 3, 4, 5])
    rps = ranked_probability_score(y_val_ord, val_proba)
    mae_xg = mean_absolute_error(y_val_home, val_xg)
    rmse_xg = rmse(y_val_home, val_xg)

    results.append({
        "n_estimators": n_est,
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll,
        "val_rps": rps,
        "val_xg_mae": mae_xg,
        "val_xg_rmse": rmse_xg
    })

    print(
        f"[{run}/{total}] "
        f"trees={n_est} depth={max_depth} lr={lr} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}, rps={rps:.4f}, xg_mae={mae_xg:.4f}"
    )

    if rps < best_rps:
        best_rps = rps
        best_params = {
            "n_estimators": n_est,
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_models = models
        best_val_proba = val_proba


results_df = pd.DataFrame(results).sort_values(["val_rps", "val_logloss", "val_xg_mae"])

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))

[1/32] trees=6000 depth=2 lr=0.05 mcw=15 gamma=4 subsample=0.8 colsample=0.8 -> acc=0.3618, logloss=1.4485, rps=0.1232, xg_mae=0.9400
[2/32] trees=6000 depth=2 lr=0.05 mcw=15 gamma=6 subsample=0.8 colsample=0.8 -> acc=0.3605, logloss=1.4483, rps=0.1232, xg_mae=0.9405
[3/32] trees=6000 depth=2 lr=0.05 mcw=15 gamma=8 subsample=0.8 colsample=0.8 -> acc=0.3566, logloss=1.4519, rps=0.1233, xg_mae=0.9408
[4/32] trees=6000 depth=2 lr=0.05 mcw=15 gamma=10 subsample=0.8 colsample=0.8 -> acc=0.3539, logloss=1.4516, rps=0.1232, xg_mae=0.9394
[5/32] trees=6000 depth=2 lr=0.05 mcw=20 gamma=4 subsample=0.8 colsample=0.8 -> acc=0.3658, logloss=1.4507, rps=0.1233, xg_mae=0.9410
[6/32] trees=6000 depth=2 lr=0.05 mcw=20 gamma=6 subsample=0.8 colsample=0.8 -> acc=0.3618, logloss=1.4497, rps=0.1232, xg_mae=0.9402
[7/32] trees=6000 depth=2 lr=0.05 mcw=20 gamma=8 subsample=0.8 colsample=0.8 -> acc=0.3632, logloss=1.4511, rps=0.1233, xg_mae=0.9403
[8/32] trees=6000 depth=2 lr=0.05 mcw=20 gamma=10 subsample=0

In [29]:

n_estimators_list = [6000]
max_depth_list = [2]
learning_rate_list = [0.05]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [10,12,14,16,18]
gamma_list = [3,4,5]

results = []

best_rps = float("inf")
best_params = None
best_models = None
best_val_proba = None

total = (
    len(n_estimators_list)
    * len(max_depth_list)
    * len(learning_rate_list)
    * len(min_child_weight_list)
    * len(gamma_list)
    * len(subsample_list)
    * len(colsample_list)
)

run = 0

for n_est, max_depth, lr, mcw, g, s, c in product(
    n_estimators_list,
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    models = {}
    val_threshold_probs = []

    # Train 5 binary models
    for k in range(5):
        clf = XGBClassifier(
            objective="binary:logistic",
            n_estimators=n_est,
            max_depth=max_depth,
            learning_rate=lr,
            subsample=s,
            colsample_bytree=c,
            min_child_weight=mcw,
            gamma=g,
            random_state=42,
            eval_metric="logloss",
            early_stopping_rounds=50,
            tree_method="hist"
        )

        clf.fit(
            X_train,
            train_targets[f"gt_{k}"],
            eval_set=[(X_val, val_targets[f"gt_{k}"])],
            verbose=False
        )

        models[k] = clf
        val_threshold_probs.append(clf.predict_proba(X_val)[:, 1])

    # Stack threshold probs
    val_threshold_probs = np.column_stack(val_threshold_probs)

    # Fix monotonicity and convert to class probs
    val_threshold_probs = enforce_monotonic_thresholds(val_threshold_probs)
    val_proba = ordinal_probs_from_thresholds(val_threshold_probs)

    # Predictions
    val_pred = np.argmax(val_proba, axis=1)
    goal_values = np.array([0, 1, 2, 3, 4, 5], dtype=float)
    val_xg = (val_proba * goal_values).sum(axis=1)

    # Metrics
    acc = accuracy_score(y_val_ord, val_pred)
    ll = log_loss(y_val_ord, val_proba, labels=[0, 1, 2, 3, 4, 5])
    rps = ranked_probability_score(y_val_ord, val_proba)
    mae_xg = mean_absolute_error(y_val_home, val_xg)
    rmse_xg = rmse(y_val_home, val_xg)

    results.append({
        "n_estimators": n_est,
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll,
        "val_rps": rps,
        "val_xg_mae": mae_xg,
        "val_xg_rmse": rmse_xg
    })

    print(
        f"[{run}/{total}] "
        f"trees={n_est} depth={max_depth} lr={lr} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}, rps={rps:.4f}, xg_mae={mae_xg:.4f}"
    )

    if rps < best_rps:
        best_rps = rps
        best_params = {
            "n_estimators": n_est,
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_models = models
        best_val_proba = val_proba


results_df = pd.DataFrame(results).sort_values(["val_rps", "val_logloss", "val_xg_mae"])

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))

[1/15] trees=6000 depth=2 lr=0.05 mcw=10 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3526, logloss=1.4487, rps=0.1232, xg_mae=0.9409
[2/15] trees=6000 depth=2 lr=0.05 mcw=10 gamma=4 subsample=0.8 colsample=0.8 -> acc=0.3553, logloss=1.4469, rps=0.1231, xg_mae=0.9402
[3/15] trees=6000 depth=2 lr=0.05 mcw=10 gamma=5 subsample=0.8 colsample=0.8 -> acc=0.3553, logloss=1.4474, rps=0.1231, xg_mae=0.9402
[4/15] trees=6000 depth=2 lr=0.05 mcw=12 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3605, logloss=1.4508, rps=0.1231, xg_mae=0.9396
[5/15] trees=6000 depth=2 lr=0.05 mcw=12 gamma=4 subsample=0.8 colsample=0.8 -> acc=0.3566, logloss=1.4499, rps=0.1232, xg_mae=0.9408
[6/15] trees=6000 depth=2 lr=0.05 mcw=12 gamma=5 subsample=0.8 colsample=0.8 -> acc=0.3592, logloss=1.4492, rps=0.1232, xg_mae=0.9406
[7/15] trees=6000 depth=2 lr=0.05 mcw=14 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3618, logloss=1.4508, rps=0.1232, xg_mae=0.9399
[8/15] trees=6000 depth=2 lr=0.05 mcw=14 gamma=4 subsample=0.8

In [30]:

n_estimators_list = [6000]
max_depth_list = [2]
learning_rate_list = [0.05]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [15,16,17]
gamma_list = [3]

results = []

best_rps = float("inf")
best_params = None
best_models = None
best_val_proba = None

total = (
    len(n_estimators_list)
    * len(max_depth_list)
    * len(learning_rate_list)
    * len(min_child_weight_list)
    * len(gamma_list)
    * len(subsample_list)
    * len(colsample_list)
)

run = 0

for n_est, max_depth, lr, mcw, g, s, c in product(
    n_estimators_list,
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    models = {}
    val_threshold_probs = []

    # Train 5 binary models
    for k in range(5):
        clf = XGBClassifier(
            objective="binary:logistic",
            n_estimators=n_est,
            max_depth=max_depth,
            learning_rate=lr,
            subsample=s,
            colsample_bytree=c,
            min_child_weight=mcw,
            gamma=g,
            random_state=42,
            eval_metric="logloss",
            early_stopping_rounds=50,
            tree_method="hist"
        )

        clf.fit(
            X_train,
            train_targets[f"gt_{k}"],
            eval_set=[(X_val, val_targets[f"gt_{k}"])],
            verbose=False
        )

        models[k] = clf
        val_threshold_probs.append(clf.predict_proba(X_val)[:, 1])

    # Stack threshold probs
    val_threshold_probs = np.column_stack(val_threshold_probs)

    # Fix monotonicity and convert to class probs
    val_threshold_probs = enforce_monotonic_thresholds(val_threshold_probs)
    val_proba = ordinal_probs_from_thresholds(val_threshold_probs)

    # Predictions
    val_pred = np.argmax(val_proba, axis=1)
    goal_values = np.array([0, 1, 2, 3, 4, 5], dtype=float)
    val_xg = (val_proba * goal_values).sum(axis=1)

    # Metrics
    acc = accuracy_score(y_val_ord, val_pred)
    ll = log_loss(y_val_ord, val_proba, labels=[0, 1, 2, 3, 4, 5])
    rps = ranked_probability_score(y_val_ord, val_proba)
    mae_xg = mean_absolute_error(y_val_home, val_xg)
    rmse_xg = rmse(y_val_home, val_xg)

    results.append({
        "n_estimators": n_est,
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll,
        "val_rps": rps,
        "val_xg_mae": mae_xg,
        "val_xg_rmse": rmse_xg
    })

    print(
        f"[{run}/{total}] "
        f"trees={n_est} depth={max_depth} lr={lr} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}, rps={rps:.4f}, xg_mae={mae_xg:.4f}"
    )

    if rps < best_rps:
        best_rps = rps
        best_params = {
            "n_estimators": n_est,
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_models = models
        best_val_proba = val_proba


results_df = pd.DataFrame(results).sort_values(["val_rps", "val_logloss", "val_xg_mae"])

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))

[1/3] trees=6000 depth=2 lr=0.05 mcw=15 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3618, logloss=1.4493, rps=0.1232, xg_mae=0.9401
[2/3] trees=6000 depth=2 lr=0.05 mcw=16 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3605, logloss=1.4479, rps=0.1231, xg_mae=0.9401
[3/3] trees=6000 depth=2 lr=0.05 mcw=17 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3605, logloss=1.4491, rps=0.1231, xg_mae=0.9402

BEST PARAMS:
{'n_estimators': 6000, 'max_depth': 2, 'learning_rate': 0.05, 'min_child_weight': 16, 'gamma': 3, 'subsample': 0.8, 'colsample_bytree': 0.8}

TOP RESULTS:
   n_estimators  max_depth  learning_rate  min_child_weight  gamma  subsample  \
1          6000          2           0.05                16      3        0.8   
2          6000          2           0.05                17      3        0.8   
0          6000          2           0.05                15      3        0.8   

   colsample_bytree  val_accuracy  val_logloss   val_rps  val_xg_mae  \
1               0.8      0.360526  

In [31]:

n_estimators_list = [6000]
max_depth_list = [2]
learning_rate_list = [0.01,0.02,0.03,0.04,0.05,0.06,0.07,0.08,0.09,0.1]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [16]
gamma_list = [3]

results = []

best_rps = float("inf")
best_params = None
best_models = None
best_val_proba = None

total = (
    len(n_estimators_list)
    * len(max_depth_list)
    * len(learning_rate_list)
    * len(min_child_weight_list)
    * len(gamma_list)
    * len(subsample_list)
    * len(colsample_list)
)

run = 0

for n_est, max_depth, lr, mcw, g, s, c in product(
    n_estimators_list,
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    models = {}
    val_threshold_probs = []

    # Train 5 binary models
    for k in range(5):
        clf = XGBClassifier(
            objective="binary:logistic",
            n_estimators=n_est,
            max_depth=max_depth,
            learning_rate=lr,
            subsample=s,
            colsample_bytree=c,
            min_child_weight=mcw,
            gamma=g,
            random_state=42,
            eval_metric="logloss",
            early_stopping_rounds=50,
            tree_method="hist"
        )

        clf.fit(
            X_train,
            train_targets[f"gt_{k}"],
            eval_set=[(X_val, val_targets[f"gt_{k}"])],
            verbose=False
        )

        models[k] = clf
        val_threshold_probs.append(clf.predict_proba(X_val)[:, 1])

    # Stack threshold probs
    val_threshold_probs = np.column_stack(val_threshold_probs)

    # Fix monotonicity and convert to class probs
    val_threshold_probs = enforce_monotonic_thresholds(val_threshold_probs)
    val_proba = ordinal_probs_from_thresholds(val_threshold_probs)

    # Predictions
    val_pred = np.argmax(val_proba, axis=1)
    goal_values = np.array([0, 1, 2, 3, 4, 5], dtype=float)
    val_xg = (val_proba * goal_values).sum(axis=1)

    # Metrics
    acc = accuracy_score(y_val_ord, val_pred)
    ll = log_loss(y_val_ord, val_proba, labels=[0, 1, 2, 3, 4, 5])
    rps = ranked_probability_score(y_val_ord, val_proba)
    mae_xg = mean_absolute_error(y_val_home, val_xg)
    rmse_xg = rmse(y_val_home, val_xg)

    results.append({
        "n_estimators": n_est,
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll,
        "val_rps": rps,
        "val_xg_mae": mae_xg,
        "val_xg_rmse": rmse_xg
    })

    print(
        f"[{run}/{total}] "
        f"trees={n_est} depth={max_depth} lr={lr} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}, rps={rps:.4f}, xg_mae={mae_xg:.4f}"
    )

    if rps < best_rps:
        best_rps = rps
        best_params = {
            "n_estimators": n_est,
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_models = models
        best_val_proba = val_proba


results_df = pd.DataFrame(results).sort_values(["val_rps", "val_logloss", "val_xg_mae"])

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))

[1/10] trees=6000 depth=2 lr=0.01 mcw=16 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3526, logloss=1.4529, rps=0.1235, xg_mae=0.9422
[2/10] trees=6000 depth=2 lr=0.02 mcw=16 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3592, logloss=1.4522, rps=0.1235, xg_mae=0.9417
[3/10] trees=6000 depth=2 lr=0.03 mcw=16 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3632, logloss=1.4523, rps=0.1233, xg_mae=0.9398
[4/10] trees=6000 depth=2 lr=0.04 mcw=16 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3658, logloss=1.4486, rps=0.1233, xg_mae=0.9407
[5/10] trees=6000 depth=2 lr=0.05 mcw=16 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3605, logloss=1.4479, rps=0.1231, xg_mae=0.9401
[6/10] trees=6000 depth=2 lr=0.06 mcw=16 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3711, logloss=1.4468, rps=0.1231, xg_mae=0.9402
[7/10] trees=6000 depth=2 lr=0.07 mcw=16 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3579, logloss=1.4545, rps=0.1233, xg_mae=0.9416
[8/10] trees=6000 depth=2 lr=0.08 mcw=16 gamma=3 subsample=0.8

In [32]:

n_estimators_list = [6000]
max_depth_list = [2]
learning_rate_list = [0.045,0.05,0.055]
subsample_list = [0.8]
colsample_list = [0.8]
min_child_weight_list = [16]
gamma_list = [3]

results = []

best_rps = float("inf")
best_params = None
best_models = None
best_val_proba = None

total = (
    len(n_estimators_list)
    * len(max_depth_list)
    * len(learning_rate_list)
    * len(min_child_weight_list)
    * len(gamma_list)
    * len(subsample_list)
    * len(colsample_list)
)

run = 0

for n_est, max_depth, lr, mcw, g, s, c in product(
    n_estimators_list,
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    models = {}
    val_threshold_probs = []

    # Train 5 binary models
    for k in range(5):
        clf = XGBClassifier(
            objective="binary:logistic",
            n_estimators=n_est,
            max_depth=max_depth,
            learning_rate=lr,
            subsample=s,
            colsample_bytree=c,
            min_child_weight=mcw,
            gamma=g,
            random_state=42,
            eval_metric="logloss",
            early_stopping_rounds=50,
            tree_method="hist"
        )

        clf.fit(
            X_train,
            train_targets[f"gt_{k}"],
            eval_set=[(X_val, val_targets[f"gt_{k}"])],
            verbose=False
        )

        models[k] = clf
        val_threshold_probs.append(clf.predict_proba(X_val)[:, 1])

    # Stack threshold probs
    val_threshold_probs = np.column_stack(val_threshold_probs)

    # Fix monotonicity and convert to class probs
    val_threshold_probs = enforce_monotonic_thresholds(val_threshold_probs)
    val_proba = ordinal_probs_from_thresholds(val_threshold_probs)

    # Predictions
    val_pred = np.argmax(val_proba, axis=1)
    goal_values = np.array([0, 1, 2, 3, 4, 5], dtype=float)
    val_xg = (val_proba * goal_values).sum(axis=1)

    # Metrics
    acc = accuracy_score(y_val_ord, val_pred)
    ll = log_loss(y_val_ord, val_proba, labels=[0, 1, 2, 3, 4, 5])
    rps = ranked_probability_score(y_val_ord, val_proba)
    mae_xg = mean_absolute_error(y_val_home, val_xg)
    rmse_xg = rmse(y_val_home, val_xg)

    results.append({
        "n_estimators": n_est,
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll,
        "val_rps": rps,
        "val_xg_mae": mae_xg,
        "val_xg_rmse": rmse_xg
    })

    print(
        f"[{run}/{total}] "
        f"trees={n_est} depth={max_depth} lr={lr} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}, rps={rps:.4f}, xg_mae={mae_xg:.4f}"
    )

    if rps < best_rps:
        best_rps = rps
        best_params = {
            "n_estimators": n_est,
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_models = models
        best_val_proba = val_proba


results_df = pd.DataFrame(results).sort_values(["val_rps", "val_logloss", "val_xg_mae"])

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))

[1/3] trees=6000 depth=2 lr=0.045 mcw=16 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3632, logloss=1.4478, rps=0.1233, xg_mae=0.9420
[2/3] trees=6000 depth=2 lr=0.05 mcw=16 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3605, logloss=1.4479, rps=0.1231, xg_mae=0.9401
[3/3] trees=6000 depth=2 lr=0.055 mcw=16 gamma=3 subsample=0.8 colsample=0.8 -> acc=0.3671, logloss=1.4527, rps=0.1231, xg_mae=0.9389

BEST PARAMS:
{'n_estimators': 6000, 'max_depth': 2, 'learning_rate': 0.055, 'min_child_weight': 16, 'gamma': 3, 'subsample': 0.8, 'colsample_bytree': 0.8}

TOP RESULTS:
   n_estimators  max_depth  learning_rate  min_child_weight  gamma  subsample  \
2          6000          2          0.055                16      3        0.8   
1          6000          2          0.050                16      3        0.8   
0          6000          2          0.045                16      3        0.8   

   colsample_bytree  val_accuracy  val_logloss   val_rps  val_xg_mae  \
2               0.8      0.36710

In [33]:

n_estimators_list = [6000]
max_depth_list = [2]
learning_rate_list = [0.055]
subsample_list = [0.6,0.7,0.8,0.9]
colsample_list = [0.6,0.7,0.8,0.9]
min_child_weight_list = [16]
gamma_list = [3]

results = []

best_rps = float("inf")
best_params = None
best_models = None
best_val_proba = None

total = (
    len(n_estimators_list)
    * len(max_depth_list)
    * len(learning_rate_list)
    * len(min_child_weight_list)
    * len(gamma_list)
    * len(subsample_list)
    * len(colsample_list)
)

run = 0

for n_est, max_depth, lr, mcw, g, s, c in product(
    n_estimators_list,
    max_depth_list,
    learning_rate_list,
    min_child_weight_list,
    gamma_list,
    subsample_list,
    colsample_list
):
    run += 1

    models = {}
    val_threshold_probs = []

    # Train 5 binary models
    for k in range(5):
        clf = XGBClassifier(
            objective="binary:logistic",
            n_estimators=n_est,
            max_depth=max_depth,
            learning_rate=lr,
            subsample=s,
            colsample_bytree=c,
            min_child_weight=mcw,
            gamma=g,
            random_state=42,
            eval_metric="logloss",
            early_stopping_rounds=50,
            tree_method="hist"
        )

        clf.fit(
            X_train,
            train_targets[f"gt_{k}"],
            eval_set=[(X_val, val_targets[f"gt_{k}"])],
            verbose=False
        )

        models[k] = clf
        val_threshold_probs.append(clf.predict_proba(X_val)[:, 1])

    # Stack threshold probs
    val_threshold_probs = np.column_stack(val_threshold_probs)

    # Fix monotonicity and convert to class probs
    val_threshold_probs = enforce_monotonic_thresholds(val_threshold_probs)
    val_proba = ordinal_probs_from_thresholds(val_threshold_probs)

    # Predictions
    val_pred = np.argmax(val_proba, axis=1)
    goal_values = np.array([0, 1, 2, 3, 4, 5], dtype=float)
    val_xg = (val_proba * goal_values).sum(axis=1)

    # Metrics
    acc = accuracy_score(y_val_ord, val_pred)
    ll = log_loss(y_val_ord, val_proba, labels=[0, 1, 2, 3, 4, 5])
    rps = ranked_probability_score(y_val_ord, val_proba)
    mae_xg = mean_absolute_error(y_val_home, val_xg)
    rmse_xg = rmse(y_val_home, val_xg)

    results.append({
        "n_estimators": n_est,
        "max_depth": max_depth,
        "learning_rate": lr,
        "min_child_weight": mcw,
        "gamma": g,
        "subsample": s,
        "colsample_bytree": c,
        "val_accuracy": acc,
        "val_logloss": ll,
        "val_rps": rps,
        "val_xg_mae": mae_xg,
        "val_xg_rmse": rmse_xg
    })

    print(
        f"[{run}/{total}] "
        f"trees={n_est} depth={max_depth} lr={lr} "
        f"mcw={mcw} gamma={g} subsample={s} colsample={c} "
        f"-> acc={acc:.4f}, logloss={ll:.4f}, rps={rps:.4f}, xg_mae={mae_xg:.4f}"
    )

    if rps < best_rps:
        best_rps = rps
        best_params = {
            "n_estimators": n_est,
            "max_depth": max_depth,
            "learning_rate": lr,
            "min_child_weight": mcw,
            "gamma": g,
            "subsample": s,
            "colsample_bytree": c
        }
        best_models = models
        best_val_proba = val_proba


results_df = pd.DataFrame(results).sort_values(["val_rps", "val_logloss", "val_xg_mae"])

print("\nBEST PARAMS:")
print(best_params)

print("\nTOP RESULTS:")
print(results_df.head(10))

[1/16] trees=6000 depth=2 lr=0.055 mcw=16 gamma=3 subsample=0.6 colsample=0.6 -> acc=0.3605, logloss=1.4524, rps=0.1234, xg_mae=0.9416
[2/16] trees=6000 depth=2 lr=0.055 mcw=16 gamma=3 subsample=0.6 colsample=0.7 -> acc=0.3658, logloss=1.4528, rps=0.1232, xg_mae=0.9408
[3/16] trees=6000 depth=2 lr=0.055 mcw=16 gamma=3 subsample=0.6 colsample=0.8 -> acc=0.3671, logloss=1.4515, rps=0.1233, xg_mae=0.9405
[4/16] trees=6000 depth=2 lr=0.055 mcw=16 gamma=3 subsample=0.6 colsample=0.9 -> acc=0.3671, logloss=1.4500, rps=0.1233, xg_mae=0.9415
[5/16] trees=6000 depth=2 lr=0.055 mcw=16 gamma=3 subsample=0.7 colsample=0.6 -> acc=0.3711, logloss=1.4517, rps=0.1234, xg_mae=0.9422
[6/16] trees=6000 depth=2 lr=0.055 mcw=16 gamma=3 subsample=0.7 colsample=0.7 -> acc=0.3658, logloss=1.4541, rps=0.1234, xg_mae=0.9418
[7/16] trees=6000 depth=2 lr=0.055 mcw=16 gamma=3 subsample=0.7 colsample=0.8 -> acc=0.3566, logloss=1.4520, rps=0.1234, xg_mae=0.9409
[8/16] trees=6000 depth=2 lr=0.055 mcw=16 gamma=3 subsa

In [ ]:

home_ordinal_models = {}
val_threshold_probs = []
test_threshold_probs = []

for k in range(5):
    print(f"Training model for P(HG > {k})")

    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=6000,
        max_depth=2,
        learning_rate=0.05,
        min_child_weight=16,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=3,
        random_state=42,
        eval_metric="logloss",
        early_stopping_rounds=50
    )

    y_train_k = train_targets[f"gt_{k}"]
    y_val_k   = val_targets[f"gt_{k}"]

    model.fit(
        X_train,
        y_train_k,
        eval_set=[(X_val, y_val_k)],
        verbose=False
    )

    home_ordinal_models[k] = model

    # Probability of class 1 = P(HG > k)
    val_prob_k = model.predict_proba(X_val)[:, 1]
    test_prob_k = model.predict_proba(X_test)[:, 1]

    val_threshold_probs.append(val_prob_k)
    test_threshold_probs.append(test_prob_k)

# Shape: (n_samples, 5)
val_threshold_probs = np.column_stack(val_threshold_probs)
test_threshold_probs = np.column_stack(test_threshold_probs)

# ---------------------------------------------------------
# 4. Enforce monotonic thresholds
# ---------------------------------------------------------

val_threshold_probs = enforce_monotonic_thresholds(val_threshold_probs)
test_threshold_probs = enforce_monotonic_thresholds(test_threshold_probs)

# ---------------------------------------------------------
# 5. Convert threshold probs -> class probs
# ---------------------------------------------------------

home_val_proba = ordinal_probs_from_thresholds(val_threshold_probs)
home_test_proba = ordinal_probs_from_thresholds(test_threshold_probs)

# Most likely class
home_val_pred_class = np.argmax(home_val_proba, axis=1)
home_test_pred_class = np.argmax(home_test_proba, axis=1)

# Expected goals from class probabilities
goal_values = np.array([0, 1, 2, 3, 4, 5], dtype=float)
home_val_pred_exp = (home_val_proba * goal_values).sum(axis=1)
home_test_pred_exp = (home_test_proba * goal_values).sum(axis=1)

# ---------------------------------------------------------
# 6. Evaluation
# ---------------------------------------------------------

print("\n===== HOME GOALS ORDINAL MODEL =====")

print("VAL ACCURACY:", accuracy_score(y_val_home_ord, home_val_pred_class))
print("TEST ACCURACY:", accuracy_score(y_test_home_ord, home_test_pred_class))

print("VAL LOGLOSS:", log_loss(y_val_home_ord, home_val_proba, labels=[0, 1, 2, 3, 4, 5]))
print("TEST LOGLOSS:", log_loss(y_test_home_ord, home_test_proba, labels=[0, 1, 2, 3, 4, 5]))

print("VAL RPS:", ranked_probability_score(y_val_home_ord, home_val_proba))
print("TEST RPS:", ranked_probability_score(y_test_home_ord, home_test_proba))

print("VAL EXPECTED-GOALS MAE:", mean_absolute_error(y_val_home, home_val_pred_exp))
print("TEST EXPECTED-GOALS MAE:", mean_absolute_error(y_test_home, home_test_pred_exp))

print("VAL EXPECTED-GOALS RMSE:", root_mean_squared_error(y_val_home, home_val_pred_exp))
print("TEST EXPECTED-GOALS RMSE:", root_mean_squared_error(y_test_home, home_test_pred_exp))

Training model for P(HG > 0)
Training model for P(HG > 1)
Training model for P(HG > 2)
Training model for P(HG > 3)
Training model for P(HG > 4)

===== HOME GOALS ORDINAL MODEL =====
VAL ACCURACY: 0.3605263157894737
TEST ACCURACY: 0.3159090909090909
VAL LOGLOSS: 1.447856555366261
TEST LOGLOSS: 1.5269340640791165
VAL RPS: 0.12312150574178676
TEST RPS: 0.1306555102956412
VAL EXPECTED-GOALS MAE: 0.9401290420044557
TEST EXPECTED-GOALS MAE: 0.9795650516541421
VAL EXPECTED-GOALS RMSE: 1.2021665218059632
TEST EXPECTED-GOALS RMSE: 1.2429209363878844


In [36]:
# =========================================================
# AWAY GOALS ORDINAL MODEL
# =========================================================
# ---------------------------------------------------------
# Prepare capped ordinal targets for AWAY goals
# ---------------------------------------------------------

y_train_away_ord = y_train_away.apply(cap_goals)
y_val_away_ord   = y_val_away.apply(cap_goals)
y_test_away_ord  = y_test_away.apply(cap_goals)

away_train_targets = make_ordinal_targets(y_train_away_ord, max_class=5)
away_val_targets   = make_ordinal_targets(y_val_away_ord, max_class=5)

away_ordinal_models = {}
val_threshold_probs = []
test_threshold_probs = []

for k in range(5):
    print(f"Training model for P(AG > {k})")

    model = XGBClassifier(
        objective="binary:logistic",
        n_estimators=6000,
        max_depth=2,
        learning_rate=0.05,
        min_child_weight=16,
        subsample=0.8,
        colsample_bytree=0.8,
        gamma=3,
        random_state=42,
        eval_metric="logloss",
        early_stopping_rounds=50
    )

    y_train_k = away_train_targets[f"gt_{k}"]
    y_val_k   = away_val_targets[f"gt_{k}"]

    model.fit(
        X_train,
        y_train_k,
        eval_set=[(X_val, y_val_k)],
        verbose=False
    )

    away_ordinal_models[k] = model

    # Probability of class 1 = P(AG > k)
    val_prob_k = model.predict_proba(X_val)[:, 1]
    test_prob_k = model.predict_proba(X_test)[:, 1]

    val_threshold_probs.append(val_prob_k)
    test_threshold_probs.append(test_prob_k)

# Shape: (n_samples, 5)
val_threshold_probs = np.column_stack(val_threshold_probs)
test_threshold_probs = np.column_stack(test_threshold_probs)

# ---------------------------------------------------------
# Enforce monotonic thresholds
# ---------------------------------------------------------

val_threshold_probs = enforce_monotonic_thresholds(val_threshold_probs)
test_threshold_probs = enforce_monotonic_thresholds(test_threshold_probs)

# ---------------------------------------------------------
# Convert threshold probs -> class probs
# ---------------------------------------------------------

away_val_proba = ordinal_probs_from_thresholds(val_threshold_probs)
away_test_proba = ordinal_probs_from_thresholds(test_threshold_probs)

# Most likely class
away_val_pred_class = np.argmax(away_val_proba, axis=1)
away_test_pred_class = np.argmax(away_test_proba, axis=1)

# Expected goals from class probabilities
goal_values = np.array([0, 1, 2, 3, 4, 5], dtype=float)
away_val_pred_exp = (away_val_proba * goal_values).sum(axis=1)
away_test_pred_exp = (away_test_proba * goal_values).sum(axis=1)

# ---------------------------------------------------------
# Evaluation
# ---------------------------------------------------------

print("\n===== AWAY GOALS ORDINAL MODEL =====")

print("VAL ACCURACY:", accuracy_score(y_val_away_ord, away_val_pred_class))
print("TEST ACCURACY:", accuracy_score(y_test_away_ord, away_test_pred_class))

print("VAL LOGLOSS:", log_loss(y_val_away_ord, away_val_proba, labels=[0, 1, 2, 3, 4, 5]))
print("TEST LOGLOSS:", log_loss(y_test_away_ord, away_test_proba, labels=[0, 1, 2, 3, 4, 5]))

print("VAL RPS:", ranked_probability_score(y_val_away_ord, away_val_proba))
print("TEST RPS:", ranked_probability_score(y_test_away_ord, away_test_proba))

print("VAL EXPECTED-GOALS MAE:", mean_absolute_error(y_val_away, away_val_pred_exp))
print("TEST EXPECTED-GOALS MAE:", mean_absolute_error(y_test_away, away_test_pred_exp))

print("VAL EXPECTED-GOALS RMSE:", root_mean_squared_error(y_val_away, away_val_pred_exp))
print("TEST EXPECTED-GOALS RMSE:", root_mean_squared_error(y_test_away, away_test_pred_exp))

Training model for P(AG > 0)
Training model for P(AG > 1)
Training model for P(AG > 2)
Training model for P(AG > 3)
Training model for P(AG > 4)

===== AWAY GOALS ORDINAL MODEL =====
VAL ACCURACY: 0.3513157894736842
TEST ACCURACY: 0.3176136363636364
VAL LOGLOSS: 1.4162863197716338
TEST LOGLOSS: 1.4734719296050611
VAL RPS: 0.11898653229138947
TEST RPS: 0.12167964116765219
VAL EXPECTED-GOALS MAE: 0.8825420007180104
TEST EXPECTED-GOALS MAE: 0.8889020735722872
VAL EXPECTED-GOALS RMSE: 1.1686127579967607
TEST EXPECTED-GOALS RMSE: 1.1542735543070262


In [ ]:
## 4.11 now I will create a matrix of scores and the probabilities for them

In [37]:
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss

# ---------------------------------------------------------
# 1. Build score matrices
# ---------------------------------------------------------

def build_score_matrix(home_proba, away_proba):
    """
    home_proba: shape (n_matches, 6)
    away_proba: shape (n_matches, 6)

    returns:
        score_matrices: shape (n_matches, 6, 6)
        where [m, i, j] = P(home=i, away=j)
    """
    n = home_proba.shape[0]
    score_matrices = np.zeros((n, 6, 6))

    for m in range(n):
        score_matrices[m] = np.outer(home_proba[m], away_proba[m])

    return score_matrices

score_test = build_score_matrix(home_test_proba, away_test_proba)
score_val = build_score_matrix(home_val_proba, away_val_proba)

In [43]:
# ---------------------------------------------------------
# 2. Derive match probabilities
# ---------------------------------------------------------

def derive_match_probs(score_matrices):
    n = score_matrices.shape[0]

    p_home_win = np.zeros(n)
    p_draw = np.zeros(n)
    p_away_win = np.zeros(n)
    p_btts = np.zeros(n)
    p_over_25 = np.zeros(n)
    p_over_15 = np.zeros(n)
    p_over_35 = np.zeros(n)

    most_likely_home = np.zeros(n, dtype=int)
    most_likely_away = np.zeros(n, dtype=int)

    for m in range(n):
        mat = score_matrices[m]

        # Home / Draw / Away
        p_home_win[m] = np.tril(mat, k=-1).sum()   # home goals > away goals
        p_draw[m]     = np.trace(mat)
        p_away_win[m] = np.triu(mat, k=1).sum()    # away goals > home goals

        # BTTS = both >= 1
        p_btts[m] = mat[1:, 1:].sum()

        # Over totals
        over15 = 0.0
        over25 = 0.0
        over35 = 0.0

        for i in range(6):
            for j in range(6):
                total = i + j

                # note: class 5 means 5+, so these are approximate
                if total > 1:
                    over15 += mat[i, j]
                if total > 2:
                    over25 += mat[i, j]
                if total > 3:
                    over35 += mat[i, j]

        p_over_15[m] = over15
        p_over_25[m] = over25
        p_over_35[m] = over35

        # Most likely score
        idx = np.unravel_index(np.argmax(mat), mat.shape)
        most_likely_home[m] = idx[0]
        most_likely_away[m] = idx[1]

    return pd.DataFrame({
        "p_home_win": p_home_win,
        "p_draw": p_draw,
        "p_away_win": p_away_win,
        "p_btts": p_btts,
        "p_over_15": p_over_15,
        "p_over_25": p_over_25,
        "p_over_35": p_over_35,
        "pred_home_score": most_likely_home,
        "pred_away_score": most_likely_away
    })

val_match_probs = derive_match_probs(score_val)
test_match_probs = derive_match_probs(score_test)
val_outcome_proba = val_match_probs[["p_home_win", "p_draw", "p_away_win"]].values
test_outcome_proba = test_match_probs[["p_home_win", "p_draw", "p_away_win"]].values

val_outcome_proba = val_outcome_proba / val_outcome_proba.sum(axis=1, keepdims=True)
test_outcome_proba = test_outcome_proba / test_outcome_proba.sum(axis=1, keepdims=True)

In [44]:
# ---------------------------------------------------------
# 3. Add expected goals and totals
# ---------------------------------------------------------

In [46]:
# ---------------------------------------------------------
# 4. True labels
# ---------------------------------------------------------

true_val_home = np.asarray(y_val_home).astype(int)
true_val_away = np.asarray(y_val_away).astype(int)

true_test_home = np.asarray(y_test_home).astype(int)
true_test_away = np.asarray(y_test_away).astype(int)

def outcome_label(h, a):
    if h > a:
        return 0   # home win
    elif h == a:
        return 1   # draw
    else:
        return 2   # away win

y_val_outcome = np.array([outcome_label(h, a) for h, a in zip(true_val_home, true_val_away)])
y_test_outcome = np.array([outcome_label(h, a) for h, a in zip(true_test_home, true_test_away)])

y_val_btts = ((true_val_home > 0) & (true_val_away > 0)).astype(int)
y_test_btts = ((true_test_home > 0) & (true_test_away > 0)).astype(int)

y_val_over25 = ((true_val_home + true_val_away) > 2).astype(int)
y_test_over25 = ((true_test_home + true_test_away) > 2).astype(int)

In [47]:
# ---------------------------------------------------------
# 5. Evaluate W/D/L probabilities
# ---------------------------------------------------------


val_outcome_proba = val_match_probs[["p_home_win", "p_draw", "p_away_win"]].values
test_outcome_proba = test_match_probs[["p_home_win", "p_draw", "p_away_win"]].values

val_outcome_pred = np.argmax(val_outcome_proba, axis=1)
test_outcome_pred = np.argmax(test_outcome_proba, axis=1)

print("\n===== MATCH-LEVEL EVALUATION =====")

print("\n--- W/D/L ---")
print("VAL ACCURACY:", accuracy_score(y_val_outcome, val_outcome_pred))
print("TEST ACCURACY:", accuracy_score(y_test_outcome, test_outcome_pred))
print("VAL LOGLOSS:", log_loss(y_val_outcome, val_outcome_proba, labels=[0,1,2]))
print("TEST LOGLOSS:", log_loss(y_test_outcome, test_outcome_proba, labels=[0,1,2]))

print("\n--- BTTS ---")
print("VAL BRIER:", brier_score_loss(y_val_btts, val_match_probs["p_btts"]))
print("TEST BRIER:", brier_score_loss(y_test_btts, test_match_probs["p_btts"]))
print("VAL ACCURACY:", accuracy_score(y_val_btts, (val_match_probs["p_btts"] >= 0.5).astype(int)))
print("TEST ACCURACY:", accuracy_score(y_test_btts, (test_match_probs["p_btts"] >= 0.5).astype(int)))

print("\n--- OVER 2.5 ---")
print("VAL BRIER:", brier_score_loss(y_val_over25, val_match_probs["p_over_25"]))
print("TEST BRIER:", brier_score_loss(y_test_over25, test_match_probs["p_over_25"]))
print("VAL ACCURACY:", accuracy_score(y_val_over25, (val_match_probs["p_over_25"] >= 0.5).astype(int)))
print("TEST ACCURACY:", accuracy_score(y_test_over25, (test_match_probs["p_over_25"] >= 0.5).astype(int)))


===== MATCH-LEVEL EVALUATION =====

--- W/D/L ---
VAL ACCURACY: 0.5092105263157894
TEST ACCURACY: 0.5465909090909091
VAL LOGLOSS: 1.002493461465012
TEST LOGLOSS: 0.9715289345505836

--- BTTS ---
VAL BRIER: 0.24765351211105732
TEST BRIER: 0.25100246451755165
VAL ACCURACY: 0.55
TEST ACCURACY: 0.5

--- OVER 2.5 ---
VAL BRIER: 0.24268830477463005
TEST BRIER: 0.2455145062575884
VAL ACCURACY: 0.5657894736842105
TEST ACCURACY: 0.5431818181818182


c:\Users\user\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:2956: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(
c:\Users\user\anaconda3\Lib\site-packages\sklearn\metrics\_classification.py:2956: UserWarning: The y_pred values do not sum to one. Make sure to pass probabilities.
  warnings.warn(


# 4.12 interpretations

the low score in bts and over 2.5 reveal that my assumption of making P(H=i,A=j)=P(H=i)P(A=j) is wrong because in football HG and AG are correlated  